# Use case: ask your codebase

You have a repo and a question like *"where is the duplicate detection implemented?"*.
`grep` needs the right identifier; an embedding needs the right meaning. A codebase question
usually needs both: exact names (`find_duplicates`, `bm25.npz`) and paraphrase ("dedupe",
"the keyword cache file").

This notebook indexes four source files of this very library, asks eight questions the way a
new contributor would, and measures `dense`, `keyword` and `hybrid` retrieval against each
other with `evaluate()`. Then a local model answers one question with citations.

**Needs:** Ollama with `nomic-embed-text`. The answer step also needs `qwen2.5:7b-instruct`.

In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STORE = ROOT / ".usecase_nb" / "codebase"          # the store lives here; delete the folder to start over
RUN_LLM = True                               # the steps that call a chat model are slow on CPU
LLM = "qwen2.5:7b-instruct"                      # follows context better than llama3.2:3b

from slim_llm_memory import topic, evaluate

FILES = ["index.py", "rerank.py", "llm.py", "evals.py"]          # ~650 lines; keeps embedding time short
code = topic("slim-llm-memory source", path=STORE)
code.add({f: (ROOT / "slim_llm_memory" / f).read_text() for f in FILES})

added 4 doc(s), 29 chunks: 0 embedded, 29 unchanged, 0 removed

`add` takes a `{name: text}` dict, so a `.py` file goes in like any other text. Chunks are split on
blank lines, which for Python means roughly one function or method per chunk. Re-run the cell:
nothing is re-embedded because the text is content-hashed.

## Eight questions, three retrieval modes

Each case is `(question, a word that must appear in the right chunk)`. Half the questions use
the exact identifier, half describe the behaviour without naming anything.

In [2]:
CASES = [
    # exact identifiers: what grep is good at
    ("where is find_duplicates implemented?",                    "def find_duplicates"),
    ("what does should_rerank check?",                           "def should_rerank"),
    ("how does validate_citations handle a dangling [n]?",       "def validate_citations"),
    ("what is REFUSAL set to?",                                  "REFUSAL ="),
    # behaviour only: what embeddings are good at
    ("how are near-duplicate items grouped into clusters?",      "def find_duplicates"),
    ("when is the cross-encoder skipped because the leader is already clear?", "relative_gap"),
    ("how does the model get told to cite the context?",         "SYSTEM_GROUNDED"),
    ("what does MRR mean in the evaluation report?",             "def mrr"),
]

reports = {mode: evaluate(code, CASES, k=5, mode=mode, min_score=0.0, label=mode)
           for mode in ["dense", "keyword", "hybrid"]}
for mode, rep in reports.items():
    print(f"{mode:<8}", rep.summary())

dense    {'hit@1': 0.5, 'hit@5': 0.875, 'mrr': 0.608}
keyword  {'hit@1': 0.625, 'hit@5': 0.875, 'mrr': 0.75}
hybrid   {'hit@1': 0.625, 'hit@5': 1.0, 'mrr': 0.781}


Read the rows together with the per-question tables below. *Dense* finds the paraphrases but
pushes two identifier questions down to rank 3 and 5, and misses the MRR question entirely.
*Keyword* puts every identifier at rank 1 and misses the one paraphrase that names nothing in
the code. *Hybrid* is the only mode that has every answer in its top 5.

In [3]:
for rep in reports.values():
    print(rep, "\n")

evaluate(dense, 8 cases, k=5):  hit@1 0.50 · hit@5 0.88 · MRR 0.61
     1  where is find_duplicates implemented?                         expects 'def find_duplicates'
     3  what does should_rerank check?                                expects 'def should_rerank'
     5  how does validate_citations handle a dangling [n]?            expects 'def validate_citations'
     1  what is REFUSAL set to?                                       expects 'REFUSAL ='
     1  how are near-duplicate items grouped into clusters?           expects 'def find_duplicates'
     3  when is the cross-encoder skipped because the leader is alre  expects 'relative_gap'
     1  how does the model get told to cite the context?              expects 'SYSTEM_GROUNDED'
     —  what does MRR mean in the evaluation report?                  expects 'def mrr' 

evaluate(keyword, 8 cases, k=5):  hit@1 0.62 · hit@5 0.88 · MRR 0.75
     1  where is find_duplicates implemented?                         expects 'def find_duplic

## Where did each hit come from?

`meta["via"]` says which leg found a chunk: `dense`, `keyword`, or `both`. Ask one question
per style and look at the labels.

In [4]:
for q in ["what does should_rerank check?", "how are near-duplicate items grouped into clusters?"]:
    r = code.ask(q, k=3, min_score=0.0)
    print(r, "\n")

ask('what does should_rerank check?')  3 hit(s) · hybrid · embed 800 ms · scan 0.29 ms
   1  0.63  rerank.py#1              … abc import ABC, abstractmethod from typing import Sequence f  [both]
   2  0.67  rerank.py#0              """Rerankers — a second, slower look at the top candidates. Re  [dense]
   3  0.64  rerank.py#2              … scores[-1] relative_gap = (gap / spread) if spread != 0 else  [dense] 



ask('how are near-duplicate items grouped into clusters?')  3 hit(s) · hybrid · embed 1122 ms · scan 0.30 ms
   1  0.69  index.py#12              … # ─── duplicates ───────────────────────────────────────────  [both]
   2  0.65  index.py#11              … != ({self.embedder.dim},)") t0 = time.time() hits = self._ra  [both]
   3  0.54  index.py#3               … def __exit__(self, *args) -> None: self.close() # ─── upsert  [both] 



## A cited answer from a local model

`answer()` hands the top chunks to a chat model and returns text that cites them by number. The
`Answer` is a `str` with `.citations` and `.hits` attached, so you can show the reader exactly
which function the claim came from.

In [5]:
if RUN_LLM:
    a = code.answer("how does find_duplicates group items, and what threshold does it use?", model=LLM, k=4)
    print(a)
    print("\ncited:", a.citations, "→", [a.hits[i - 1].id for i in a.citations])

The function `find_duplicates` clusters open items by cosine similarity ≥ 0.86 [1].

cited: [1] → ['index.py#12']


## Takeaways

- **Index code as `{path: text}`.** No parser needed; blank-line chunking lands on function boundaries.
- **Use `hybrid` for code.** Identifiers are rare tokens that BM25 nails and embeddings blur.
- **Check `via`** when a result surprises you. It tells you whether to tune the keyword side or the dense side.
- **Scale it up** by adding the whole package: `code.add({p.relative_to(ROOT).as_posix(): p.read_text() for p in ROOT.rglob("*.py")})`.
  Only new or changed files are embedded on the next run.

In [6]:
code.close()